# agri-agronomy-core quickstart

Minimal example: load the bundled synthetic sample pack, resolve a
taxon by name, and run one irrigation calculation.

No network access. No external files. Deterministic.

In [ ]:
from pathlib import Path

from agri_core import registry
from agri_core.irrigation import irrigate_field_etc
from agri_core.pack import load_knowledge_set


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found above: ' + str(start))


repo_root = _find_repo_root(Path.cwd())
sample_pack = repo_root / 'src' / 'agri_core' / '_samples' / 'sample_pack'
pack = load_knowledge_set(sample_pack, allow_synthetic=True)

info = registry.meta(pack)
print(f'pack version: {info.pack_version}')
print(f'content hash: {info.content_hash[:24]}...')
print(f'engine:       {info.engine_version}')

In [ ]:
outcome = registry.resolve_taxon(pack, 'Testus annualis')
assert outcome.status == 'ok', f'expected ok, got {outcome.status}'
taxon_id = outcome.taxon_id
print(f'resolved taxon: {taxon_id}')

In [ ]:
result = irrigate_field_etc(
    pack,
    taxon_id,
    phase='midseason',
    et0_mm_day=5.0,
)
assert result.status == 'ok', f'expected ok, got {result.status}'
print(f'net irrigation: {result.value:.3f} {result.unit}')
print(f'fact IDs used:  {result.provenance.fact_ids}')